In [10]:
# mypy: ignore-errors
# ruff: noqa
import gc
import math
import os
import sys
import warnings

import numpy as np
import optuna
import pandas as pd
import torch
from tqdm import tqdm

warnings.simplefilter("ignore")
from sklearn.model_selection import KFold

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from drGAT import drGAT
from drGAT.load_data import load_data
from drGAT.sampler import NewSampler
from get_params import get_params
from metrics import compute_metrics_stats

name = "nci"
# task = "cell"
task = "drug"

method = "GAT"
target_dim = [
    1,  # Drug
    # 0  # Cell
]

PATH = f"../{name}_data/"

(
    res,
    pos_num,
    null_mask,
    S_d,
    S_c,
    S_g,
    A_cg,
    A_dg,
    _,
    _,
    _,
) = load_data(name)
res = res.T
cell_sum = np.sum(res, axis=1)
drug_sum = np.sum(res, axis=0)

load nci
unique drugs: 177
unique genes: 251
DTI unique genes:  251
Top 90% variable genes:  2383
Total:  2582
Final gene exp shape: (60, 2582)
Final drug Act shape: (1005, 60)


100%|██████████| 25/25 [00:02<00:00,  9.27it/s]


Done!


In [2]:
def drGAT_new(
    res_mat,
    null_mask,
    target_dim,
    target_index,
    S_d,
    S_c,
    S_g,
    A_cg,
    A_dg,
    PATH,
    params,
    device,
    seed,
):
    sampler = NewSampler(
        res_mat,
        null_mask,
        target_dim,
        target_index,
        S_d,
        S_c,
        S_g,
        A_cg,
        A_dg,
        PATH,
        seed,
    )

    (_, _, _, best_val_labels, best_val_prob, best_metrics, _, _, _) = drGAT.train(
        sampler, params=params, device=device, verbose=False
    )

    return best_val_labels, best_val_prob

In [3]:
def objective(trial):
    params = {
        "n_drug": S_d.shape[0],
        "n_cell": S_c.shape[0],
        "n_gene": S_g.shape[0],
        "dropout1": trial.suggest_float("dropout1", 0.1, 0.5, step=0.05),
        "dropout2": trial.suggest_float("dropout2", 0.1, 0.5, step=0.05),
        "hidden1": trial.suggest_int("hidden1", 256, 1024),
        "hidden2": trial.suggest_int("hidden2", 64, min(512, trial.params["hidden1"])),
        "hidden3": trial.suggest_int("hidden3", 32, min(256, trial.params["hidden2"])),
        "epochs": 100,
        # trial.suggest_int("epochs", 100, 10000, step=100),
        "heads": trial.suggest_int("heads", 2, 8),
        "activation": trial.suggest_categorical("activation", ["relu", "gelu"]),
        "optimizer": trial.suggest_categorical("optimizer", ["Adam", "AdamW"]),
        "lr": trial.suggest_float("lr", 1e-5, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        "scheduler": trial.suggest_categorical("scheduler", [None, "Cosine"]),
        "gnn_layer": method,
    }

    # スケジューラ関連パラメータの条件付き追加
    if params["scheduler"] == "Cosine":
        # T_maxの最小値を1以上に保証
        min_epoch_div = max(1, params["epochs"] // 5)  # 最小値1を強制
        max_epoch_div = max(
            min_epoch_div + 1, params["epochs"] // 2
        )  # low < highを保証

        params["T_max"] = trial.suggest_int(
            "T_max", low=min_epoch_div, high=max_epoch_div
        )

        # 追加のチェック（防御的プログラミング）
        if params["T_max"] <= 0:
            raise optuna.TrialPruned(f"Invalid T_max: {params['T_max']}")

    try:
        n_kfold = 1
        true_datas = pd.DataFrame()
        predict_datas = pd.DataFrame()
        for dim in target_dim:
            for seed, target_index in tqdm(enumerate(np.arange(res.shape[dim]))):
                if dim:
                    if drug_sum[target_index] < 10:
                        continue
                else:
                    if cell_sum[target_index] < 10:
                        continue

                for fold in range(n_kfold):
                    true_data, predict_data = drGAT_new(
                        res_mat=res,
                        null_mask=null_mask.T.values,
                        target_dim=dim,
                        target_index=target_index,
                        S_d=S_d,
                        S_c=S_c,
                        S_g=S_g,
                        A_cg=A_cg,
                        A_dg=A_dg,
                        PATH=PATH,
                        params=params,
                        device=device,
                        seed=seed,
                    )

                    true_datas = pd.concat(
                        [true_datas, pd.DataFrame(true_data).T], ignore_index=True
                    )
                    predict_datas = pd.concat(
                        [predict_datas, pd.DataFrame(predict_data).T], ignore_index=True
                    )

        metrics_result = compute_metrics_stats(
            trial=trial,
            true=true_datas,
            pred=predict_datas,
            target_metrics=["AUROC", "AUPR", "F1", "ACC"],
        )

        return tuple(metrics_result["target_values"])

    except RuntimeError as e:
        if "CUDA out of memory" in str(e):
            print(f"Pruned trial {trial.number}: CUDA OOM")

            with torch.cuda.device("cuda"):
                torch.cuda.empty_cache()
            gc.collect()

            raise optuna.TrialPruned(f"OOM at trial {trial.number}")
        else:
            print(f"RuntimeError in trial {trial.number}: {str(e)}")
            raise e

    except ValueError as e:
        if "Input contains NaN" in str(e):
            print(f"Pruned trial {trial.number}: Input contains NaN")
            raise optuna.TrialPruned(f"NaN input at trial {trial.number}")
        else:
            print(f"ValueError in trial {trial.number}: {str(e)}")
            raise e

    except ZeroDivisionError:
        print(f"Pruned trial {trial.number}: ZeroDivisionError in CosineAnnealingLR")
        raise optuna.TrialPruned("ZeroDivisionError in CosineAnnealingLR")
    except Exception as e:
        print(f"Unexpected error in trial {trial.number}: {str(e)}")
        raise e

In [8]:
study = optuna.create_study(
    directions=["maximize"] * 4,
    sampler=optuna.samplers.NSGAIISampler(),
    pruner=optuna.pruners.HyperbandPruner(),
    storage=f"sqlite:///{method}_{task}_small.sqlite3",
    study_name=method,
    load_if_exists=True,
)
study.optimize(objective, n_trials=1000)

[I 2025-04-12 17:40:09,153] Using an existing study with name 'GAT' instead of creating a new one.
0it [00:00, ?it/s]

Using device: cuda


0it [00:03, ?it/s]
[I 2025-04-12 17:40:13,069] Trial 2 pruned. OOM at trial 2


Pruned trial 2: CUDA OOM


0it [00:00, ?it/s]

...

0it [00:00, ?it/s]

Using device: cuda


1it [01:15, 75.22s/it]

Best model found at epoch 100
Using device: cuda


1it [01:28, 88.26s/it]
[W 2025-04-12 17:41:56,733] Trial 9 failed with parameters: {'dropout1': 0.1, 'dropout2': 0.45000000000000007, 'hidden1': 388, 'hidden2': 66, 'hidden3': 36, 'heads': 5, 'activation': 'gelu', 'optimizer': 'AdamW', 'lr': 0.00045323918487636967, 'weight_decay': 5.12384338625353e-06, 'scheduler': 'Cosine', 'T_max': 35} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/optuna/study/_optimize.py", line 196, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_18748/4054493943.py", line 52, in objective
    true_data, predict_data = drGAT_new(
  File "/tmp/ipykernel_18748/4265841952.py", line 30, in drGAT_new
    (_, _, _, best_val_labels, best_val_prob, best_metrics, _, _, _) = drGAT.train(
  File "/spin1/home/linux/inouey2/drGAT/drGAT/drGAT.py", line 284, in train
    train_attention = train_one_epoch(
  File "/spin1/home/linux/inouey2/drGAT/drGA

In [13]:
n_kfold = 1
true_datas = pd.DataFrame()
predict_datas = pd.DataFrame()
for dim in target_dim:
    for seed, target_index in tqdm(enumerate(np.arange(res.shape[dim]))):
        break

0it [00:00, ?it/s]


In [14]:
cell_sum[target_index]

456

In [15]:
drug_sum[target_index]

KeyError: 0

In [16]:
drug_sum

740       38
752       35
755       34
757       39
762       26
          ..
811429    25
812926    16
812927    23
813488    24
820919    25
Length: 1005, dtype: int64

In [17]:
cell_sum

786_0          456
A498           593
A549           453
ACHN           532
BT_549         323
CAKI_1         567
CCRF_CEM       618
COLO205        499
DU_145         383
EKVX           172
HCC_2998       239
HCT_116        654
HCT_15         345
HL_60          641
HOP_62         452
HOP_92         586
HS578T         412
HT29           528
IGROV1         432
KM12           434
K_562          499
LOXIMVI        704
M14            497
MALME_3M       408
MCF7           715
MDA_MB_231     258
MDA_MB_435     466
MDA_N           82
MOLT_4         620
NCI_ADR_RES    252
NCI_H226       355
NCI_H23        445
NCI_H322M      241
NCI_H460       613
NCI_H522       576
OVCAR_3        318
OVCAR_4        221
OVCAR_5        204
OVCAR_8        360
PC_3           356
RPMI_8226      444
RXF_393        592
SF_268         375
SF_295         586
SF_539         615
SK_MEL_2       297
SK_MEL_28      223
SK_MEL_5       539
SK_OV_3        389
SN12C          486
SNB_19         280
SNB_75         666
SR          